# Smart MCQ Solver

## Submission By:

- **Name:** Varun Agnihotri
- **Roll Number:** `24f2004142`

In [ ]:
import os
import random
import sys
from pathlib import Path

import pandas as pd
import torch

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IS_KAGGLE:
    print("Running in Kaggle environment.")
    TEST_CSV_PATH = Path(
        "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
    )
    SUBMISSION_OUT_PATH = Path("submission.csv")

    # Resolve Model Registry
    base_model_dir = Path(
        "/kaggle/input/models/pythonicvarun/smart-mcq-solver/pytorch/local"
    )
    if base_model_dir.exists():
        version_dirs = list(base_model_dir.glob("*"))
        MODEL_DIR = version_dirs[0] if version_dirs else base_model_dir
    else:
        MODEL_DIR = Path(
            "/kaggle/input/models/pythonicvarun/smart-mcq-solver/pytorch/local/2"
        )

    SRC_PATH = MODEL_DIR / "src"
else:
    print("Running in local environment.")
    if Path(os.getcwd()).name == "notebooks":
        os.chdir("..")

    print(f"Current working directory: {os.getcwd()}")

    TEST_CSV_PATH = Path("dataset/test.csv")
    MODEL_DIR = Path("outputs/local")
    SRC_PATH = Path("src")
    SUBMISSION_OUT_PATH = MODEL_DIR / "submission_local_model.csv"

# add source path to python search path
if SRC_PATH.exists():
    if str(SRC_PATH.resolve()) not in sys.path:
        sys.path.append(str(SRC_PATH.resolve()))
    print(f"Added source path to sys.path: {SRC_PATH.resolve()}")
else:
    print(
        f"WARNING: Source path '{SRC_PATH}' not found. Make sure packages are importable."  # noqa: E501
    )

print(f"Test CSV path: {TEST_CSV_PATH}")
print(f"Model directory: {MODEL_DIR}")
print(f"Submission output path: {SUBMISSION_OUT_PATH}")

In [ ]:
from local import config

# override config directories
config.OUTPUT_DIR = MODEL_DIR
config.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device in use: {config.DEVICE}")

In [ ]:
from local.inference import load_artifacts, predict

# Check test file
if not TEST_CSV_PATH.exists():
    print("Required test dataset not found, exiting!")
    exit(1)

test_df = pd.read_csv(TEST_CSV_PATH)
print(f"Loaded test dataset with {len(test_df)} records.")

In [ ]:
# Load model, vocab, and retriever
model, vocab, retriever = load_artifacts()

model

In [ ]:
# Run prediction pipeline
sub_df = predict(test_df, model, retriever, vocab, is_train=False)

# Ensure directory exists for output if running locally
SUBMISSION_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

sub_df.to_csv(SUBMISSION_OUT_PATH, index=False)
print(f"Saved submission to: {SUBMISSION_OUT_PATH.resolve()}")

In [ ]:
sub_df.head(10)

In [ ]:
# Sample Visualization
random.seed(42)

sample_indices = random.sample(range(len(test_df)), min(3, len(test_df)))

for idx in sample_indices:
    row = test_df.iloc[idx]
    pred = sub_df.iloc[idx]["Prediction"]

    print("-" * 120)
    print(f"[ID {row['id']}] PROMPT:")
    print(row["prompt"])
    print("-" * 60)
    for c in ["A", "B", "C", "D", "E"]:
        print(f"({c}) {row[c]}")
    print("-" * 60)

    ctx = retriever.retrieve(row["prompt"], top_k=1)
    print(f"Retrieved Context (Top-1): {ctx[:300]}...")
    print("-" * 60)
    print(f"Prediction (MAP@3 Order): {pred}")
    print("-" * 120)
    print()